# Classification of odontogenic pathologies — protocol reconstruction






A CSV `manifest.csv` must contain `image_path,case_id,label,magnification`. `case_id` must identify the same patient across all their images and magnifications; if a patient has several specimens, they must share the identifier. Labels: AM, DC, OKC, OOC, HDF. Magnifications: 4, 10, 20, 40, 60.



## 1. Set up Colab
Select a GPU runtime. Run the installation if dependencies are missing; restart the runtime if you upgrade PyTorch. Pretrained weights are downloaded from the official source the first time. The effective versions and the hardware are recorded in the results.


In [ ]:
import importlib.util, subprocess, sys
required = {'torch':'torch', 'torchvision':'torchvision', 'numpy':'numpy',
            'pandas':'pandas', 'sklearn':'scikit-learn', 'scipy':'scipy',
            'PIL':'Pillow', 'matplotlib':'matplotlib', 'joblib':'joblib'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: set the local paths in the next cell.')


## 2. Configuration
Change the paths. The original folders are used only to generate an inventory: afterwards you must fill in the case identifiers. For 20 repetitions, change `SEEDS` to `list(range(20))`


In [ ]:
from pathlib import Path
import os, json, random, hashlib, platform, copy, gc, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy
from scipy.stats import t
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, log_loss, roc_auc_score
import joblib
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
import matplotlib.pyplot as plt

DATA_ROOT = Path('/content/drive/MyDrive/GorlinIA/A_D_K_CROPPED')
MANIFEST_PATH = DATA_ROOT / 'manifest.csv'
OUTPUT_ROOT = Path('/content/drive/MyDrive/GorlinIA/resultados_reconstruccion')
SEEDS = list(range(5))
MAGNIFICATIONS = [4, 10, 20, 40, 60]
ARCHITECTURES = ['densenet121', 'alexnet']
CLASSES = ['AM', 'DC', 'OKC', 'OOC', 'HDF']
LABEL_TO_ID = {label: i for i, label in enumerate(CLASSES)}
FOLDER_TO_LABEL = {'AMELOBLASTOMA':'AM', 'DENTIGEROUS':'DC', 'KERATOCYST':'OKC',
                   'ORTHOKERATINIZED':'OOC', 'DENTAL':'HDF'}
# Confirm that DENTAL really is hyperplastic dental follicle (HDF).
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 32, 30, 5
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
PCA_RANDOM_STATE, PCA_DDOF, PCA_THRESHOLD = 42, 0, 1.5
RUN_ABLATION = True
NUM_WORKERS = 0  # simplifies reproducibility in Colab
BOOTSTRAP_REPLICATES = 2000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    print('A GPU is recommended to run all the experiments.')


## 3. Create a template if you do not have the CSV yet



In [ ]:
def inventory_template(root):
    records = []
    for folder, label in FOLDER_TO_LABEL.items():
        for mag in MAGNIFICATIONS:
            directory = root / folder / f'{mag}X'
            if not directory.is_dir():
                raise FileNotFoundError(f'{directory} does not exist; fix DATA_ROOT or the folder names.')
            for p in sorted(directory.rglob('*')):
                if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}:
                    records.append({'image_path':str(p.resolve()), 'case_id':'',
                                    'label':label, 'magnification':mag})
    if not records:
        raise ValueError('No patches were found.')
    return pd.DataFrame(records)

if not MANIFEST_PATH.exists():
    template = inventory_template(DATA_ROOT)
    template_path = DATA_ROOT / 'manifest_template.csv'
    if not template_path.exists():
        template.to_csv(template_path, index=False)
    raise RuntimeError(f'Fill in case_id in {template_path} and save it as {MANIFEST_PATH}.')
print('CSV found:', MANIFEST_PATH)


## 4. Validate the dataset and record its identity



In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def validate_manifest(csv_path):
    df = pd.read_csv(csv_path, dtype={'image_path':str,'case_id':str,'label':str})
    needed = {'image_path','case_id','label','magnification'}
    if not needed <= set(df.columns):
        raise ValueError(f'Required columns: {sorted(needed)}')
    df = df[list(sorted(needed))].copy()
    if df.isna().any().any():
        raise ValueError('There are empty fields; fill in all case_id values and metadata.')
    for c in ['image_path','case_id','label']:
        df[c] = df[c].str.strip()
        if (df[c] == '').any():
            raise ValueError(f'Empty values in {c}.')
    df['label'] = df['label'].str.upper()
    if not set(df.label) <= set(CLASSES):
        raise ValueError('Unknown labels: ' + str(set(df.label)-set(CLASSES)))
    mags = pd.to_numeric(df.magnification, errors='raise')
    if not np.all(mags == np.floor(mags)) or set(mags.astype(int)) != set(MAGNIFICATIONS):
        raise ValueError('magnification must contain the integers 4, 10, 20, 40 and 60.')
    df['magnification'] = mags.astype(int)
    df['image_path'] = df.image_path.map(lambda p: str((Path(p) if Path(p).is_absolute() else csv_path.parent / p).resolve()))
    if df.image_path.duplicated().any():
        raise ValueError('An image path appears more than once.')
    if (df.groupby('case_id').label.nunique() != 1).any():
        raise ValueError('A case_id has several labels; review the identification.')
    coverage = df.groupby('case_id').magnification.apply(set)
    if not coverage.map(lambda s: s == set(MAGNIFICATIONS)).all():
        raise ValueError('There are cases without all five magnifications; review the inventory.')
    cases = df[['case_id','label']].drop_duplicates()
    counts = cases.label.value_counts()
    if set(counts.index) != set(CLASSES) or counts.min() < 5:
        raise ValueError('At least five cases per class are required to attempt the stratified split.')
    hashes, pixel_hashes = [], []
    for p in df.image_path:
        if not Path(p).is_file():
            raise FileNotFoundError(p)
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert('RGB')
            if im.size != (512,512):
                raise ValueError(f'{p}: size {im.size}; 512x512 patches are required.')
            pixel_hashes.append(hashlib.sha256(im.tobytes()).hexdigest())
        hashes.append(file_sha256(p))
    df['sha256'] = hashes
    df['pixel_sha256'] = pixel_hashes
    if (df.groupby('pixel_sha256').case_id.nunique() > 1).any():
        raise ValueError('Identical content assigned to different cases: possible duplication across patients.')
    df['label_id'] = df.label.map(LABEL_TO_ID)
    return df.sort_values(['case_id','magnification','image_path']).reset_index(drop=True)

manifest = validate_manifest(MANIFEST_PATH)
print('Unique cases per class (paper: AM16, DC18, OKC20, OOC19, HDF17):')
print(manifest[['case_id','label']].drop_duplicates().groupby('label').size())
print('Images per magnification/class (paper: 235 in each cell):')
print(pd.crosstab(manifest.magnification, manifest.label))
print('Total:', len(manifest), 'images;', manifest.case_id.nunique(), 'cases')


## 5. Patient-level splits, reproducibility and study directory
The split is generated **only once per seed** and reused for all models and magnifications. The actual numbers of cases are saved; rounding may make the proportions differ slightly from the nominal 60/20/20. Each execution of this cell opens a new folder; to resume without mixing experiments, keep `RUN_DIR` and the cells in memory.


In [ ]:
def seed_everything(seed):
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

def split_cases(df, seed):
    cases = df[['case_id','label']].drop_duplicates().sort_values('case_id').reset_index(drop=True)
    train, rest = train_test_split(cases, test_size=0.4, random_state=seed, stratify=cases.label)
    val, test = train_test_split(rest, test_size=0.5, random_state=seed, stratify=rest.label)
    groups = {'train':train, 'validation':val, 'test':test}
    for name, part in groups.items():
        if set(part.label) != set(CLASSES):
            raise ValueError(f'Classes missing in {name}, seed {seed}.')
    if set(train.case_id) & set(val.case_id) or set(train.case_id) & set(test.case_id) or set(val.case_id) & set(test.case_id):
        raise AssertionError('Case overlap within a repetition.')
    return pd.concat([p.assign(split=name) for name,p in groups.items()], ignore_index=True)

RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('reconstruccion_%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
manifest.to_csv(RUN_DIR/'manifest_validated.csv', index=False)
config = dict(seeds=SEEDS, magnifications=MAGNIFICATIONS, architectures=ARCHITECTURES,
              classes=CLASSES, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
              patience=PATIENCE, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
              pca_state=PCA_RANDOM_STATE, pca_ddof=PCA_DDOF, pca_threshold=PCA_THRESHOLD,
              run_ablation=RUN_ABLATION, pretrained_trainable='final_layer_only',
              provenance='reconstruction_not_original_code', input='unfiltered_512px_RGB_patches')
(RUN_DIR/'config.json').write_text(json.dumps(config,indent=2))
environment = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
                   numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__,
                   device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
(RUN_DIR/'environment.json').write_text(json.dumps(environment,indent=2))
CASE_SPLITS = {}
for seed in SEEDS:
    split = split_cases(manifest, seed)
    CASE_SPLITS[seed] = split
    split.to_csv(RUN_DIR/f'case_split_seed_{seed}.csv',index=False)
    print('Seed',seed, split.groupby('split').size().to_dict())
print('Results:', RUN_DIR)


## 6. PCA filtering on the training set only


In [ ]:
def pca_filter_training(train_df, audit_dir):
    if set(train_df.split) != {'train'}:
        raise ValueError('PCA only accepts training rows.')
    audit_dir.mkdir(parents=True, exist_ok=True)
    retained, audit = [], []
    for label in CLASSES:
        group = train_df[train_df.label == label].copy()
        if len(group) < 3:
            raise ValueError(f'Fewer than three images for PCA of {label}.')
        pixels = []
        for path in group.image_path:
            with Image.open(path) as im:
                pixels.append(np.asarray(ImageOps.exif_transpose(im).convert('L'), dtype=np.float32).reshape(-1)/255.0)
        X = np.stack(pixels)
        pca = PCA(n_components=2, svd_solver='randomized', random_state=PCA_RANDOM_STATE)
        scores = pca.fit_transform(X)
        center, sd = scores.mean(axis=0), scores.std(axis=0, ddof=PCA_DDOF)
        lower, upper = center-PCA_THRESHOLD*sd, center+PCA_THRESHOLD*sd
        keep = ((scores >= lower) & (scores <= upper)).all(axis=1)
        if keep.sum() == 0:
            raise ValueError(f'PCA removed all images of {label}; review the data.')
        record = group[['image_path','case_id','label','magnification']].copy()
        record['pc1'], record['pc2'], record['keep'] = scores[:,0], scores[:,1], keep
        audit.append(record)
        retained.append(group.loc[keep])
        joblib.dump(dict(pca=pca, lower=lower, upper=upper, ddof=PCA_DDOF),audit_dir/f'pca_{label}.joblib')
        del X, pixels, pca
        gc.collect()
    pd.concat(audit).to_csv(audit_dir/'pca_training_audit.csv',index=False)
    result = pd.concat(retained).sort_values('image_path').reset_index(drop=True)
    print('PCA:',len(train_df)-len(result),'excluded out of',len(train_df),'training images')
    return result


## 7. Models and transforms



In [ ]:
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD = [0.229,0.224,0.225]
def image_transform(training):
    ops = [transforms.Resize((224,224), interpolation=transforms.InterpolationMode.BILINEAR)]
    if training:
        ops += [transforms.RandomRotation(20, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.RandomHorizontalFlip(p=0.5)]
    return transforms.Compose(ops+[transforms.ToTensor(),transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])

class PatchDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.transform = image_transform(training)
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as im:
            image = ImageOps.exif_transpose(im).convert('RGB')
            image = self.transform(image)
        return image, int(row.label_id)

def make_model(architecture, pretrained=True):
    if architecture == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        for p in model.parameters(): p.requires_grad_(False)
        model.classifier = nn.Linear(model.classifier.in_features,len(CLASSES))
    elif architecture == 'alexnet':
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None)
        for p in model.parameters(): p.requires_grad_(False)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features,len(CLASSES))
    else:
        raise ValueError(architecture)
    return model.to(DEVICE)

def loader(frame, training, seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(PatchDataset(frame,training),batch_size=BATCH_SIZE,shuffle=training,
                      generator=generator,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')

@torch.no_grad()
def evaluate(model, data_loader):
    model.eval()
    total_loss, total = 0.0, 0
    probs, labels = [], []
    for images,y in data_loader:
        images,y = images.to(DEVICE),y.to(DEVICE)
        logits = model(images)
        total_loss += nn.functional.cross_entropy(logits,y,reduction='sum').item()
        total += len(y)
        probs.append(logits.softmax(1).cpu().numpy())
        labels.append(y.cpu().numpy())
    return total_loss/total, np.concatenate(labels), np.concatenate(probs)


## 8. Metrics and training



In [ ]:
def classification_metrics(y, probs):
    y = np.asarray(y, dtype=int)
    probs = np.asarray(probs)
    pred = probs.argmax(1)
    cm = confusion_matrix(y,pred,labels=range(len(CLASSES)))
    tp = np.diag(cm).astype(float)
    fn, fp = cm.sum(1)-tp, cm.sum(0)-tp
    tn = cm.sum()-tp-fn-fp
    sensitivity = np.divide(tp,tp+fn,out=np.full(len(CLASSES),np.nan),where=(tp+fn)>0)
    specificity = np.divide(tn,tn+fp,out=np.full(len(CLASSES),np.nan),where=(tn+fp)>0)
    class_f1 = f1_score(y,pred,labels=range(len(CLASSES)),average=None,zero_division=0)
    auc = []
    for c in range(len(CLASSES)):
        binary = (y==c).astype(int)
        auc.append(float(roc_auc_score(binary,probs[:,c])) if len(np.unique(binary))==2 else np.nan)
    overall = dict(accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(np.mean(sensitivity)),
                   sensitivity=float(np.mean(sensitivity)),specificity=float(np.mean(specificity)),
                   f1=float(np.mean(class_f1)),roc_auc=float(np.mean(auc)),
                   log_loss=float(log_loss(y,probs,labels=list(range(len(CLASSES))))))
    classwise = pd.DataFrame({'class':CLASSES,'sensitivity':sensitivity,'specificity':specificity,
                              'f1':class_f1,'roc_auc':auc,'support_images':cm.sum(1)})
    return overall,classwise,cm

def train_and_test(train_df,val_df,test_df,architecture,seed,out_dir):
    out_dir.mkdir(parents=True,exist_ok=False)
    seed_everything(seed)
    for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
        frame.to_csv(out_dir/f'{name}_images.csv',index=False)
    model = make_model(architecture)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                                 lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    train_loader = loader(train_df,True,seed)
    val_loader = loader(val_df,False,seed)
    best_loss, waiting, best_epoch = float('inf'),0,None
    history = []
    for epoch in range(1,MAX_EPOCHS+1):
        model.train()
        model.features.eval()  # also freezes BatchNorm statistics
        loss_sum,correct,n = 0.,0,0
        for images,y in train_loader:
            images,y = images.to(DEVICE),y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = nn.functional.cross_entropy(logits,y)
            if not torch.isfinite(loss): raise RuntimeError('Non-finite loss.')
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()*len(y)
            correct += (logits.argmax(1)==y).sum().item()
            n += len(y)
        val_loss, val_y, val_p = evaluate(model,val_loader)
        row = dict(epoch=epoch,train_loss=loss_sum/n,train_accuracy=correct/n,
                   validation_loss=val_loss,validation_accuracy=float((val_p.argmax(1)==val_y).mean()))
        history.append(row)
        pd.DataFrame(history).to_csv(out_dir/'history.csv',index=False)
        print(f'{architecture} seed={seed} epoch={epoch}: train={row["train_loss"]:.4f} val={val_loss:.4f}')
        if val_loss < best_loss:
            best_loss,best_epoch,waiting = val_loss,epoch,0
            state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            torch.save(dict(state_dict=state,architecture=architecture,seed=int(seed),
                            epoch=epoch,classes=CLASSES),out_dir/'best_model.pt')
            del state
        else:
            waiting += 1
            if waiting >= PATIENCE: break
    saved = torch.load(out_dir/'best_model.pt',map_location=DEVICE,weights_only=True)
    model.load_state_dict(saved['state_dict'])
    del saved
    test_loss,y,probs = evaluate(model,loader(test_df,False,seed))
    metrics,classwise,cm = classification_metrics(y,probs)
    metrics.update(best_epoch=int(best_epoch),test_cross_entropy=float(test_loss),
                   train_images=len(train_df),validation_images=len(val_df),test_images=len(test_df),
                   train_cases=int(train_df.case_id.nunique()),test_cases=int(test_df.case_id.nunique()))
    predictions = test_df[['image_path','case_id','label','label_id','magnification']].reset_index(drop=True).copy()
    if not np.array_equal(predictions.label_id.to_numpy(),y): raise AssertionError('Incorrect prediction order.')
    predictions['predicted_label'] = [CLASSES[k] for k in probs.argmax(1)]
    for i,c in enumerate(CLASSES): predictions[f'p_{c}'] = probs[:,i]
    predictions.to_csv(out_dir/'test_predictions.csv',index=False)
    classwise.to_csv(out_dir/'classwise_metrics.csv',index=False)
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(out_dir/'confusion_matrix.csv')
    (out_dir/'metrics.json').write_text(json.dumps(metrics,indent=2))
    fig,axes = plt.subplots(1,2,figsize=(10,3.5))
    h=pd.DataFrame(history)
    for ax,key in zip(axes,['loss','accuracy']):
        ax.plot(h.epoch,h['train_'+key],label='Training')
        ax.plot(h.epoch,h['validation_'+key],label='Validation')
        ax.set(xlabel='Epoch',ylabel=key); ax.legend()
    fig.tight_layout();fig.savefig(out_dir/'learning_curves.png',dpi=160);plt.close(fig)
    del model,optimizer,train_loader,val_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metrics


## 9. Run the study
This cell starts real training runs: by default 50 main configurations and 5 ablation runs. Switching to 20 seeds produces 200 + 20 training runs.


In [ ]:
def run_experiments():
    results = []
    for seed in SEEDS:
        mapping = CASE_SPLITS[seed][['case_id','split']]
        joined = manifest.merge(mapping,on='case_id',validate='many_to_one')
        for mag in MAGNIFICATIONS:
            data = joined[joined.magnification==mag].copy()
            tr = data[data.split=='train'].sort_values('image_path').reset_index(drop=True)
            va = data[data.split=='validation'].sort_values('image_path').reset_index(drop=True)
            te = data[data.split=='test'].sort_values('image_path').reset_index(drop=True)
            pca_dir = RUN_DIR/f'seed_{seed}'/f'mag_{mag}'/'pca'
            audit_file = pca_dir/'pca_training_audit.csv'
            if audit_file.exists():
                audit = pd.read_csv(audit_file)
                kept_paths = set(audit.loc[audit.keep==True,'image_path'])
                tr_pca = tr[tr.image_path.isin(kept_paths)].reset_index(drop=True)
            else:
                tr_pca = pca_filter_training(tr,pca_dir)
            configs = [(arch,True) for arch in ARCHITECTURES]
            if RUN_ABLATION and mag==40: configs.append(('densenet121',False))
            for arch,use_pca in configs:
                directory = RUN_DIR/f'seed_{seed}'/f'mag_{mag}'/f'{arch}_pca_{int(use_pca)}'
                metric_file = directory/'metrics.json'
                if metric_file.exists():
                    metrics = json.loads(metric_file.read_text())
                else:
                    metrics = train_and_test(tr_pca if use_pca else tr,va,te,arch,seed,directory)
                results.append(dict(seed=seed,magnification=mag,architecture=arch,pca=use_pca,**metrics))
                pd.DataFrame(results).to_csv(RUN_DIR/'metrics_by_run.csv',index=False)
    return pd.DataFrame(results)

results = run_experiments()
print('Study finished:',RUN_DIR)


## 10. Summary and exploratory comparisons
The tables are computed exclusively from the previous runs. The sample SD (`ddof=1`) is used to describe the variability between repetitions. The Nadeau–Bengio correction uses the nominal test/training ratio of 0.20/0.60, and the intervals are **pointwise**, not simultaneous.

In [ ]:
from itertools import combinations
METRIC_NAMES=['accuracy','balanced_accuracy','log_loss','sensitivity','specificity','f1','roc_auc']
summary = results.groupby(['architecture','magnification','pca'])[METRIC_NAMES].agg(['mean','std'])
summary.to_csv(RUN_DIR/'summary_mean_sample_sd.csv')
print(summary)

def holm_adjust(pvalues):
    p=np.asarray(pvalues,float); order=np.argsort(p); adjusted=np.empty(len(p)); previous=0.
    for rank,index in enumerate(order):
        previous=max(previous,min(1.,(len(p)-rank)*p[index]))
        adjusted[index]=previous
    return adjusted

def corrected_difference(values_a, values_b, ratio=0.20/0.60):
    d=np.asarray(values_a,float)-np.asarray(values_b,float)
    if len(d)<2 or not np.isfinite(d).all(): raise ValueError('At least two valid pairs are required.')
    mean=float(d.mean()); se=float(np.sqrt((1/len(d)+ratio)*d.var(ddof=1)))
    if se==0:
        return dict(difference_pp=mean,ci_low_pp=mean,ci_high_pp=mean,p_value=np.nan,
                    note='Zero variance of differences: t-test not estimable')
    half=float(t.ppf(.975,len(d)-1)*se)
    return dict(difference_pp=mean,ci_low_pp=mean-half,ci_high_pp=mean+half,
                p_value=float(2*t.sf(abs(mean/se),len(d)-1)),note='exploratory')

def add_holm(table):
    if table.p_value.isna().any():
        table['p_holm']=np.nan
        warnings.warn('Degenerate test: Holm is not computed for this family.')
    else: table['p_holm']=holm_adjust(table.p_value)
    return table

main=results[results.pca==True]
comparison_tables=[]
for arch in ARCHITECTURES:
    pivot=main[main.architecture==arch].pivot(index='seed',columns='magnification',values='accuracy').sort_index()
    if pivot.isna().any().any() or len(pivot)!=len(SEEDS): raise ValueError('Incomplete experiments.')
    rows=[]
    for a,b in combinations(MAGNIFICATIONS,2):
        rows.append(dict(architecture=arch,first_magnification=b,second_magnification=a,
                         **corrected_difference(100*pivot[b],100*pivot[a])))
    comparison_tables.append(add_holm(pd.DataFrame(rows)))
comparisons=pd.concat(comparison_tables,ignore_index=True)
comparisons.to_csv(RUN_DIR/'magnification_all_comparisons.csv',index=False)

rows=[]
for mag in MAGNIFICATIONS:
    pivot=main[main.magnification==mag].pivot(index='seed',columns='architecture',values='accuracy').sort_index()
    rows.append(dict(magnification=mag,**corrected_difference(100*pivot.densenet121,100*pivot.alexnet)))
architecture_comparisons=add_holm(pd.DataFrame(rows))
architecture_comparisons.to_csv(RUN_DIR/'architecture_comparisons.csv',index=False)

if RUN_ABLATION:
    ab=results[(results.architecture=='densenet121') & (results.magnification==40)]
    ablation_rows=[]
    for metric in METRIC_NAMES:
        pivot=ab.pivot(index='seed',columns='pca',values=metric).sort_index()
        ablation_rows.append(dict(metric=metric,with_pca_mean=pivot[True].mean(),with_pca_sd=pivot[True].std(),
                                 without_pca_mean=pivot[False].mean(),without_pca_sd=pivot[False].std(),
                                 mean_difference=(pivot[True]-pivot[False]).mean()))
    pd.DataFrame(ablation_rows).to_csv(RUN_DIR/'pca_ablation_summary.csv',index=False)
print(architecture_comparisons)


## 11. Case-level bootstrap for a single run (optional)



In [ ]:
def bootstrap_one_run(predictions_path, n_boot=2000, random_state=12345):
    df=pd.read_csv(predictions_path,dtype={'case_id':str})
    grouped=df.groupby('case_id',sort=True)
    if (grouped.label.nunique()!=1).any(): raise ValueError('Contradictory labels within a case.')
    blocks={case:idx.to_numpy() for case,idx in grouped.groups.items()}
    cases=df[['case_id','label']].drop_duplicates()
    by_class={label:cases.loc[cases.label==label,'case_id'].to_numpy() for label in CLASSES}
    if any(len(ids)==0 for ids in by_class.values()): raise ValueError('Classes missing in test set.')
    y=df.label_id.to_numpy();p=df[[f'p_{c}' for c in CLASSES]].to_numpy()
    point=classification_metrics(y,p)[0]
    rng=np.random.default_rng(random_state);samples=[]
    for _ in range(n_boot):
        selected=np.concatenate([rng.choice(ids,size=len(ids),replace=True) for ids in by_class.values()])
        idx=np.concatenate([blocks[case] for case in selected])
        samples.append(classification_metrics(y[idx],p[idx])[0])
    boot=pd.DataFrame(samples)
    return pd.DataFrame([dict(metric=m,estimate=point[m],ci_low=boot[m].quantile(.025),
                              ci_high=boot[m].quantile(.975)) for m in METRIC_NAMES])

# Optional example after training:
# path = RUN_DIR/'seed_0'/'mag_40'/'densenet121_pca_1'/'test_predictions.csv'
# bootstrap_one_run(path, BOOTSTRAP_REPLICATES).to_csv(path.parent/'case_bootstrap_conditional.csv',index=False)


## 12. Checks before using the results in the article
1. Review `manifest_validated.csv` and the real patient identifiers.
2. Confirm that the input patches had no prior filtering or augmentation, and review the reconstruction decisions listed at the beginning.
3. Keep `config.json`, `environment.json`, all splits and predictions, PCA audits and checkpoints.
4. Update tables and figures with the results obtained, even if they differ from the previous ones; do not replace historical files without documenting what changed.
5. Describe this notebook as a reconstructed implementation until its correspondence with the study's runs is verified. Publish an identifiable version and cite the DOI of that version, where applicable.


